### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [9]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    raise RuntimeError("GROQ_API_KEY is not set. Set it in your environment or .env before running this cell.")

os.environ["GROQ_API_KEY"] = groq_key

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
print(getattr(response, 'content', response))

<think>
Okay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. Parrots are known for their ability to mimic human speech, right? But why do they do that? I need to break this down.

First, I remember that some birds like parrots, mynah birds, and others have this ability called vocal learning. But why have they evolved this trait? Maybe it's related to communication in the wild. In the wild, parrots might use calls to communicate with their flock. So maybe talking in captivity is an extension of that natural behavior.

Also, I've heard that parrots are highly social animals. They live in groups in the wild, so communication is important for bonding, finding food, or avoiding predators. In captivity, they might direct their vocalizations towards their human caretakers as a way to socialize. So talking could be their way of interacting with humans.

Another angle is that talking serves a purpose in the wild, like identifying each other or coord

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

# Binding tool with model
model_with_tools=model.bind_tools([get_weather])

In [11]:
# Calling the tool
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. So I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'm4s4e1xnt', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 154, 'total_tokens': 241, 'completion_time': 0.14585206, 'completion_tokens_details': {'reasoning_tokens': 63}, 'prompt_time': 0.012817419, 'prompt_tokens_details': None, 'queue_time': 0.15999762, 'total_time': 0.158669479}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e6abb-56af-7872-

### Tool Execution Loops

In [12]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. Let me know if you need more details! ☀️


In [13]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': '25v7s4mby', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 153, 'total_tokens': 239, 'completion_time': 0.150407762, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006561911, 'prompt_tokens_details': None, 'queue_time': 0.062916399, 'total_time': 0.156969673}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'l